#### DQ Check: gold_daily_revenue

**Production purpose:** Validate the Gold layer after each pipeline run.  
Fails the notebook task — and therefore the job — if critical assertions fail.\
It should also trigger the job-level failure notification automatically.

**Task name in job:** `dq_check_gold`  
**Depends on:** `run_sdp_pipeline`

In [0]:
dbutils.widgets.text("run_date", "2024-01-15", "Run Date")

run_date = dbutils.widgets.get("run_date")
table   = "dev.dbx_course.gold_daily_revenue"

print(f"Running DQ check for run_date={run_date}")

df = spark.table(table).filter(f"order_date = '{run_date}'")    

# Check 1: table is not empty
row_count = df.count()
assert row_count > 0, f"FAIL: {table} has 0 rows"
print(f"PASS: row_count = {row_count}")


# Check 2: no null revenue
null_revenue = df.filter("total_revenue IS NULL").count()
assert null_revenue == 0, f"FAIL: {null_revenue} rows with null revenue"
print(f"PASS: null_revenue = {null_revenue}")


# Check 3: no negative revenue
neg_revenue = df.filter("total_revenue < 0").count()
assert neg_revenue == 0, f"FAIL: {neg_revenue} rows with negative revenue"
print(f"PASS: neg_revenue = {neg_revenue}")

# Publish row_count for downstream tasks
dbutils.jobs.taskValues.set(key="row_count", value=row_count)

print(f"\nAll DQ checks passed for {run_date}. {row_count} rows validated.")